# Getting started

This is the smallest complete JuPedSim simulation: build a room, place agents,
send them to an exit, and watch them leave. Every later notebook in this
*Fundamentals* series zooms in on one piece of what you see here.

See {doc}`Object model & lifecycle </concepts/object_model>` for how the
pieces fit together.

## The pieces

A simulation is assembled from a handful of objects:

- **Geometry** — the walkable area (a `shapely` polygon).
- **Model** — the operational model that moves agents (here the Collision-Free Speed Model).
- **Simulation** — ties geometry, model, and trajectory output together.
- **Stages** — destinations agents head to; here a single **exit**.
- **Journey** — the route through one or more stages.
- **Agents** — the pedestrians, placed in the geometry.

In [ ]:
import shapely

# A 10 m x 10 m room.
geometry = shapely.Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])

import matplotlib.pyplot as plt
from pedpy import WalkableArea, plot_walkable_area

plot_walkable_area(walkable_area=WalkableArea(geometry)).set_aspect("equal")
plt.show()

In [ ]:
import pathlib

import jupedsim as jps

trajectory_file = pathlib.Path("getting_started.sqlite")
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(output_file=trajectory_file),
)

# One exit on the right wall, reached via a single-stage journey.
exit_id = simulation.add_exit_stage([(9, 4), (10, 4), (10, 6), (9, 6)])
journey_id = simulation.add_journey(jps.JourneyDescription([exit_id]))

# Place 20 agents on the left side of the room.
n_agents = 20
positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (3, 0.5), (3, 9.5), (0.5, 9.5)]),
    number_of_agents=n_agents,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=exit_id,
            position=position,
            radius=0.12,
        )
    )

# Iterate until the room is empty.
while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    simulation.iterate()

print(f"Evacuated {n_agents} agents in {simulation.iteration_count()} iterations "
      f"({simulation.elapsed_time():.1f} s).")

## Watch it

Read the recorded trajectories back and animate them.

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Try one change

Re-run the notebook after changing `n_agents` to `50`, or swap
`jps.CollisionFreeSpeedModel()` for another model (see
{doc}`Operational models </notebooks/fundamentals/02_models>`). Watch how the evacuation time changes.

## See also

- {doc}`Geometry </notebooks/fundamentals/01_geometry>` — walkable areas and obstacles.
- {doc}`Operational models </notebooks/fundamentals/02_models>` — the models that move agents.
- {doc}`Object model & lifecycle </concepts/object_model>`.
- To scale a study into parameter sweeps and Monte-Carlo runs, see
  [jupedsim-scenarios](https://scenarios.jupedsim.org).